# 02 · Data sources (skeleton)

Loaders in `pavana_ml.data.*` are **stubs** that raise `NotImplementedError`. Implement them,
save raw pulls to `ml/data/raw/<source>/`, and document provenance in `docs/data-sources.md`.

| Source | Use | Notes |
|---|---|---|
| NASA FIRMS | hotspots (lat, lon, time, FRP) | needs `FIRMS_MAP_KEY`; archive vs NRT |
| Open-Meteo weather | wind, RH, precip | request `wind_speed_unit=ms`, `timezone=GMT` |
| Open-Meteo air quality | CAMS PM2.5 baseline | need **issue-time** forecasts for a fair baseline |
| OpenAQ | station PM2.5 (target) | sparse in Kalimantan; needs `OPENAQ_API_KEY` |

In [ ]:
from pavana_ml.config import get_settings

settings = get_settings()
# Study area (Kubu Raya region) — adjust once stations are identified.
BBOX = (108.8, -1.2, 110.4, 0.6)  # west, south, east, north
START, END = "2023-08-01", "2023-10-31"  # example dry-season window; verify data availability

In [ ]:
from pavana_ml.data import firms

if settings.firms_map_key:
    try:
        df_fire = firms.fetch_area(
            settings.firms_map_key, "VIIRS_SNPP_SP", BBOX, day_range=10, date=START
        )
    except NotImplementedError as e:
        print("TODO:", e)
else:
    print("FIRMS_MAP_KEY not set — skipping")

In [ ]:
from pavana_ml.data import open_meteo, openaq

for name, fn in [
    ("weather", lambda: open_meteo.fetch_weather_hourly(-0.1, 109.38, START, END)),
    ("air quality", lambda: open_meteo.fetch_air_quality_hourly(-0.1, 109.38, START, END)),
    # TODO: replace 0 with a real OpenAQ location id near the study area
    ("openaq", lambda: openaq.fetch_station_pm25(settings.openaq_api_key or "", 0, START, END)),
]:
    try:
        fn()
    except NotImplementedError as e:
        print(f"TODO ({name}):", e)